# Stock Scout AI — V1

## M1 + M2 + Persistent Paper Portfolio

**S&P 500 → Data Engine → Scanner → Technical Engine → Top 10 → Persistent Paper Portfolio**

La pipeline rimane deterministica e tracciabile. Il paper portfolio ora persiste posizioni, trade, P&L e snapshot in SQLite.


## 1. Setup Colab


In [ ]:
!rm -rf /content/stock-scout-ai
!git clone https://github.com/freshfrisk666-creator/stock-scout-ai.git /content/stock-scout-ai
%cd /content/stock-scout-ai
!python -m pip install --upgrade pip -q
!pip -q install -r requirements.txt


## 2. Import e configurazione


In [ ]:
from pathlib import Path
import pandas as pd

from main import load_config, run_pipeline

cfg = load_config()
print('Repository:', Path.cwd())
print('History period:', cfg['market']['history_period'])
print('Portfolio capital:', cfg['portfolio']['starting_cash'])
print('Max positions:', cfg['portfolio']['positions'])


## 3. Esecuzione completa della V1


In [ ]:
(
    scan,
    top10,
    open_positions,
    snapshot,
    portfolio_status,
    closed,
) = run_pipeline(cfg)

print(f'Scanned symbols passing liquidity filter: {len(scan)}')
print(f'Top 10 rows: {len(top10)}')
print(f'Open paper positions: {len(open_positions)}')
print(f'Closed this run: {len(closed)}')


## 4. Top 10 — output principale


In [ ]:
display(
    top10[
        ['rank', 'ticker', 'technical_score', 'entry', 'stop', 'target', 'risk_reward']
    ].round(2)
)


## 5. Stato del paper portfolio

Le posizioni vengono mantenute tra le run tramite SQLite. Un ticker già OPEN non viene duplicato. I posti liberati da un'uscita possono essere riempiti dal Top 10 successivo.


In [ ]:
if portfolio_status.empty:
    print('No open positions.')
else:
    display(
        portfolio_status[
            [
                'ticker', 'entry_price', 'shares', 'notional',
                'stop', 'target', 'mark_price',
                'market_value', 'unrealized_pnl'
            ]
        ].round(2)
    )

print('\nPortfolio snapshot:')
print(snapshot)


## 6. Trade chiusi in questa run


In [ ]:
if closed:
    display(pd.DataFrame(closed).round(2))
else:
    print('No positions closed in this run.')


## 7. Database persistente

Tabelle principali:

- `scans`
- `technical_signals`
- `top10`
- `portfolio_state`
- `positions`
- `trades`
- `portfolio_snapshots`


In [ ]:
from database.db import get_connection

db_path = cfg['database']['path']
with get_connection(db_path) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        conn,
    )

display(tables)
print('SQLite DB:', db_path)


## V1 checkpoint

La V1 è ora composta da un motore tecnico deterministico e da un paper portfolio persistente: OPEN/CLOSED, P&L realizzato/non realizzato, storico BUY/SELL e snapshot di equity.
